# Train


In [1]:
%pip install torch torchvision scikit-learn matplotlib Pillow

Note: you may need to restart the kernel to use updated packages.


## Paramètres de ce notebook

In [ ]:
from pathlib import Path
from datetime import datetime

random_state = 42
data_loc = Path("data_nouveau_split")
dossier_train = data_loc / "train"
dossier_val = data_loc / "val"
dossier_test = data_loc / "test"
epochs_phase1 = 8
epochs_phase2 = 2

## CPU/GPU

In [39]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('Using gpu: %s ' % torch.cuda.is_available())

Using gpu: False 


## Préparer les images pour EfficientNet

On redimensionne les images à 224 × 224 et on applique la normalisation attendue par les poids ImageNet. (Source: documentation efficientnetb0 / transformation v2)

In [40]:
from torchvision.transforms import v2

transform = v2.Compose([
    v2.ToImage(),
    v2.Resize((224, 224), antialias=True),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

## Chargement des datasets

ImageFolder lit les images en RGB et attribue une classe à chaque sous-dossier (source: documentation imagefolder)

In [41]:
from torchvision.datasets import ImageFolder

train_data = ImageFolder(dossier_train, transform=transform)
val_data = ImageFolder(dossier_val, transform=transform)
test_data = ImageFolder(dossier_test, transform=transform)

classes = ["NORMAL", "PNEUMONIA"]

#verif
for dataset in (train_data, val_data, test_data):
    print(dataset.class_to_idx == {"NORMAL": 0, "PNEUMONIA": 1})


True
True
True


##  Création des batchs 


In [42]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

## Modèle

In [53]:
from torch import nn
import torchvision.models as models

weights = models.EfficientNet_B0_Weights.DEFAULT # poids pré entraînés sur imageNet

model = models.efficientnet_b0(weights=weights)

model.classifier = nn.Sequential( #nouvelle tete
    nn.Linear(1280, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, 1),
)

model = model.to(device)

for parameter in model.features.parameters():
    parameter.requires_grad = False # On gèle les poids du modèle pré-entraîné

## Loss

Utilisation deBCEWithLogitsLoss car elle combine sigmoid et perte binaire de façon numériquement stable.

In [44]:
criterion = nn.BCEWithLogitsLoss()

def classification_loss(model, logits, labels):
    loss = criterion(logits, labels)
    return loss + 1e-4 * model.classifier[0].weight.square().sum()

## Evaluation

In [ ]:
def test_CNN(modele, batchs_test, critere):  

    modele.eval()  
    test_loss = 0  
    correct = 0  

    with torch.no_grad():  #désactiver le calcul des gradients 

        for data, target in batchs_test : 
            data = data.to(device)  
            target = target.to(device) 

            output = modele(data).flatten()
            test_loss += critere(output, target.float()).item()*len(target) 

            pred = torch.sigmoid(output) >= 0.5  #prédit pneumonia si la probabilité atteint 0,5
            correct += pred.eq(target).sum()  

        test_loss = test_loss /len(batchs_test.dataset) 
        print('\nTest set: Avg. loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(test_loss, correct, len(batchs_test.dataset), 100. * correct / len(batchs_test.dataset)))
        return test_loss

## Train

In [ ]:
from tqdm.auto import tqdm #viauslalisation de la progression

def train_CNN(model, trainloader, val_loader, loss_fn, n_epochs, optimizer):

    meilleure_loss = test_CNN(model, val_loader, loss_fn)
    torch.save(model.state_dict(), "meilleur_modele.pth")

    for epoch in tqdm(range(n_epochs), desc="Progression totale"):

        model.train()
        model.features.eval()

        running_loss = 0.0
        for batch_idx, (inputs, targets) in enumerate(tqdm(trainloader, desc=f"Époque {epoch + 1}/{n_epochs}", leave=False)):

            optimizer.zero_grad()
            inputs = inputs.to(device)
            targets = targets.to(device) 
            outputs = model(inputs).flatten()

            loss = loss_fn(outputs, targets.float())
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(targets)

        val_loss = test_CNN(model, val_loader, loss_fn)

        if val_loss < meilleure_loss:
            meilleure_loss = val_loss
            torch.save(model.state_dict(), "meilleur_modele.pth")

    print(f'epoch : {epoch + 1} loss: {running_loss / len(trainloader.dataset):.3f}')
    model.load_state_dict(torch.load("meilleur_modele.pth", map_location=device, weights_only=True))

## Train the head

In [47]:
optimizer = torch.optim.Adam(model.classifier.parameters(),lr=0.001)
train_CNN(model,train_loader,val_loader,criterion,epochs_phase1,optimizer)


Test set: Avg. loss: 0.6693, Accuracy: 682/874 (78%)



Progression totale:   0%|          | 0/10 [00:00<?, ?it/s]

Époque 1/10:   0%|          | 0/128 [00:00<?, ?it/s]


Test set: Avg. loss: 0.0884, Accuracy: 845/874 (97%)



Époque 2/10:   0%|          | 0/128 [00:00<?, ?it/s]

...


Test set: Avg. loss: 0.1151, Accuracy: 838/874 (96%)



Époque 9/10:   0%|          | 0/128 [00:00<?, ?it/s]


Test set: Avg. loss: 0.0666, Accuracy: 854/874 (98%)



Époque 10/10:   0%|          | 0/128 [00:00<?, ?it/s]


Test set: Avg. loss: 0.0666, Accuracy: 850/874 (97%)

epoch : 10 loss: 0.093


##  Adjust last blocs

On dégèle les derniers blocs de features et on réduit le taux d’apprentissage.

In [49]:
for parameter in model.features[-2:].parameters():
    parameter.requires_grad = True

optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),lr=0.00001)

train_CNN(model, train_loader, val_loader, criterion, epochs_phase2, optimizer)


Test set: Avg. loss: 0.0666, Accuracy: 854/874 (98%)



Progression totale:   0%|          | 0/2 [00:00<?, ?it/s]

Époque 1/2:   0%|          | 0/128 [00:00<?, ?it/s]


Test set: Avg. loss: 0.0579, Accuracy: 854/874 (98%)



Époque 2/2:   0%|          | 0/128 [00:00<?, ?it/s]


Test set: Avg. loss: 0.0602, Accuracy: 855/874 (98%)

epoch : 2 loss: 0.080


## Compare thresholds

Comparaison des seuils de 0,3 à 0,8 

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

def evaluate(modele, batchs_test, critere, seuils=(0.3, 0.4, 0.5, 0.6, 0.7, 0.8)):
    modele.eval()
    test_loss = 0
    labels = []
    probabilites = []

#collecte des labels et probabilités 
    with torch.no_grad():
        for data, target in batchs_test:
            data = data.to(device)
            target = target.to(device)

            output = modele(data).flatten()
            test_loss += critere(output, target.float()).item() * len(target)

            labels.extend(target.cpu().tolist())
            probabilites.extend(torch.sigmoid(output).cpu().tolist())

#loss et AUC moyenne sur l'ensemble du test
    test_loss = test_loss / len(batchs_test.dataset)
    print(f"Loss : {test_loss:.4f}") 
    print(f"AUC : {roc_auc_score(labels, probabilites):.4f}")

#affichage des metriques dependant du seuil
    for seuil in seuils:
        pred = [int(p >= seuil) for p in probabilites]
        print(f"\nSeuil : {seuil}")
        print(classification_report(labels, pred, labels=[0, 1], target_names=["NORMAL", "PNEUMONIA"], digits=4, zero_division=0))

evaluate(model, val_loader, criterion)

## Final evaluation on test set

In [ ]:
evaluate(model, test_loader, criterion, seuils=)